In [0]:
# ============================================================
# 01_bronze_ingestion
# Healthcare Data Warehouse | Medallion Architecture
# Layer: Bronze (Raw Ingestion)
# Description: Reads 9 CSVs from ADLS Gen2 landing container,
#              adds audit columns, writes Delta tables to bronze
# ============================================================
 
#  Configuration 
storage_account_name = "adlshospitaldwh"
container_landing    = "landing"
container_bronze     = "bronze"
container_silver     = "silver"
container_gold       = "gold"
 
application_id = "ad3497a3-a8be-4251-8e01-4b7bc1c781b4"
tenant_id      = "2be11f7b-4e63-4e64-b8a4-992816fe3359"
client_secret  = "********"   
 
# Service Principal OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
LANDING_PATH = f"abfss://{container_landing}@{storage_account_name}.dfs.core.windows.net/"
BRONZE_PATH  = f"abfss://{container_bronze}@{storage_account_name}.dfs.core.windows.net/"
SILVER_PATH  = f"abfss://{container_silver}@{storage_account_name}.dfs.core.windows.net/"
GOLD_PATH    = f"abfss://{container_gold}@{storage_account_name}.dfs.core.windows.net/"
 
print(" Configuration complete")
print(f"   Landing : {LANDING_PATH}")
print(f"   Bronze  : {BRONZE_PATH}")
print(f"   Silver  : {SILVER_PATH}")
print(f"   Gold    : {GOLD_PATH}")
 

 Configuration complete
   Landing : abfss://landing@adlshospitaldwh.dfs.core.windows.net/
   Bronze  : abfss://bronze@adlshospitaldwh.dfs.core.windows.net/
   Silver  : abfss://silver@adlshospitaldwh.dfs.core.windows.net/
   Gold    : abfss://gold@adlshospitaldwh.dfs.core.windows.net/


In [0]:
# Datawarehousing - Bronze Ingestion 
from pyspark.sql.functions import current_timestamp, input_file_name, lit
 
tables = [
    "patients",
    "encounters",
    "diagnoses",
    "claims_and_billing",
    "denials",
    "procedures",
    "medications",
    "providers",
    "lab_tests",
]
 
success = []
failed  = []
 
for table in tables:
    try:
        print(f" Ingesting {table}...")
 
        df = (spark.read.format("csv")
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .load(f"{LANDING_PATH}{table}.csv"))
 
        df_bronze = (df
                     .withColumn("_ingested_at", current_timestamp())
                     .withColumn("_source_file", input_file_name())
                     .withColumn("_layer",       lit("bronze")))
 
        (df_bronze.write
                  .format("delta")
                  .mode("overwrite")
                  .save(f"{BRONZE_PATH}{table}"))
 
        row_count = df_bronze.count()
        print(f" {table}: {row_count:,} rows written to bronze")
        success.append(table)
 
    except Exception as e:
        print(f" FAILED — {table}: {e}")
        failed.append(table)
 
print("\n" + "=" * 45)
print(f" Succeeded : {len(success)}/9 tables — {success}")
if failed:
    print(f" Failed    : {failed}")
else:
    print(" Bronze ingestion complete — no errors!")
print("=" * 45)
 
 


 Ingesting patients...
 patients: 60,000 rows written to bronze
 Ingesting encounters...
 encounters: 70,000 rows written to bronze
 Ingesting diagnoses...
 diagnoses: 70,000 rows written to bronze
 Ingesting claims_and_billing...
 claims_and_billing: 70,000 rows written to bronze
 Ingesting denials...
 denials: 5,998 rows written to bronze
 Ingesting procedures...
 procedures: 126,021 rows written to bronze
 Ingesting medications...
 medications: 94,498 rows written to bronze
 Ingesting providers...
 providers: 1,491 rows written to bronze
 Ingesting lab_tests...
 lab_tests: 54,537 rows written to bronze

 Succeeded : 9/9 tables — ['patients', 'encounters', 'diagnoses', 'claims_and_billing', 'denials', 'procedures', 'medications', 'providers', 'lab_tests']
 Bronze ingestion complete — no errors!


In [0]:
# Cell 2: Verification 
print("\n" + "=" * 45)
print("BRONZE LAYER VERIFICATION")
print("=" * 45)
 
expected_counts = {
    "patients":           60_000,
    "encounters":         70_000,
    "diagnoses":          70_000,
    "claims_and_billing": 70_000,
    "denials":             5_998,
    "procedures":        126_021,
    "medications":        94_498,
    "providers":           1_491,
    "lab_tests":          54_537,
}
 
for table in tables:
    try:
        df = spark.read.format("delta").load(f"{BRONZE_PATH}{table}")
        actual = df.count()
        expected = expected_counts[table]
        status = "" if actual == expected else " COUNT MISMATCH"
        print(f"{status} {table}: {actual:,} rows | {len(df.columns)} cols | expected {expected:,}")
    except Exception as e:
        print(f" Could not read {table}: {e}")
 
print("=" * 45)


BRONZE LAYER VERIFICATION
 patients: 60,000 rows | 19 cols | expected 60,000
 encounters: 70,000 rows | 16 cols | expected 70,000
 diagnoses: 70,000 rows | 9 cols | expected 70,000
 claims_and_billing: 70,000 rows | 14 cols | expected 70,000
 denials: 5,998 rows | 13 cols | expected 5,998
 procedures: 126,021 rows | 10 cols | expected 126,021
 medications: 94,498 rows | 13 cols | expected 94,498
 providers: 1,491 rows | 13 cols | expected 1,491
 lab_tests: 54,537 rows | 13 cols | expected 54,537
